In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import queer
from queer.arpes import arpes_mesh, arpes_path
from queer.momentum import (
    sweep_binding_energy_surfaces,
    save_surface_plots,
    fermi_surface_points,
)
from queer.paths import data_path, results_path
from queer.spectral import (
    spectral_along_path,
    spectral_arpes_path_sweep,
    plot_spectral_path,
)

In [ ]:
# Input files
file_path = data_path("materials", "ag_primitive")
hr = "wannier90_hr.dat"
model = queer.model(path=file_path, ef=8.310342, poscar='POSCAR')
g_vec = model.g_vec

In [ ]:
k_points = 200
band_path = model.kpath("GAMMA-X-W-K-GAMMA-L", k_points)
bands, ax = model.plot_band_path(band_path, ylim=(-4, 8))

In [ ]:
k_points = 200
band_path = model.kpath("L-W-K-L", k_points)
bands, ax = model.plot_band_path(band_path, ylim=(-7, -2))

In [ ]:
N = 300
factor = 4
binding_range = [3.2, 6.2]
binding_step = 0.5
photon_energy = 21.2
fermi_energy = 0
V0 = 15
binding_energy = 0
align = [[0, 0, 1], [1, 1, 1]]

In [ ]:
mesh_array = np.array(
    arpes_mesh(photon_energy, fermi_energy, binding_energy, V0, N, align=align)
)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(mesh_array.T[0], mesh_array.T[1], mesh_array.T[2])
plt.show()

In [ ]:
fig_bz, ax_bz = model.plot_brillouin_zone(
    mesh=mesh_array,
    align_dir=[1, 1, 1],
    mesh_clip_bz=True,
    mesh_style="wireframe",
)
plt.show()

In [ ]:
mask, kx, ky = fermi_surface_points(mesh_array, model, g_vec, delta=0.05)
plt.scatter(kx, ky, s=6, alpha=0.9, c='red')
plt.xlabel(r"$k_x$ [$\AA^{-1}$]")
plt.ylabel(r"$k_y$ [$\AA^{-1}$]")
plt.show()

In [ ]:
N = 100
factor = 6
binding_range = [4.7, 6.7]
binding_step = 0.5
photon_energy = 21.2
fermi_energy = 0
V0 = [10, 11, 12]
align = [[0, 0, 1], [1, 1, 1]]
energy_shift = [0.15, 0.2, 0.25]

In [ ]:
results = sweep_binding_energy_surfaces(
    model, g_vec,
    photon_energy, fermi_energy,
    binding_range, binding_step,
    v0_list=V0, n_points=N, factor=factor,
    align=align, energy_shifts=energy_shift,
)
for (v, shift), surfaces in results.items():
    save_surface_plots(
        surfaces,
        results_path("ag_primitive", "figs", f"V0_{v}_shift_{shift}"),
        v0=v, energy_shift=shift,
        rotate_from=[0, 0, 1], rotate_to=[1, 1, 1],
        style="intensity",
    )

In [ ]:
N = 100
factor = 4
binding_range = [4.7, 6.7]
binding_step = 0.5
V0 = [12]
energy_shift = [0.25]
results_white = sweep_binding_energy_surfaces(
    model, g_vec,
    photon_energy, fermi_energy,
    binding_range, binding_step,
    v0_list=V0, n_points=N, factor=factor,
    align=align, energy_shifts=energy_shift,
)
for (v, shift), surfaces in results_white.items():
    save_surface_plots(
        surfaces,
        results_path("ag_primitive", "figs", f"V0_{v}_shift_{shift}"),
        v0=v, energy_shift=shift,
        rotate_from=[0, 0, 1], rotate_to=[1, 1, 1],
        style="scatter_white", tol=0.1,
    )

In [ ]:
for (v, shift), surfaces in results_white.items():
    save_surface_plots(
        surfaces,
        results_path("ag_primitive", "figs", f"V0_{v}_shift_{shift}"),
        v0=v, energy_shift=shift,
        rotate_from=[0, 0, 1], rotate_to=[1, 1, 1],
        style="scatter_black_transparent", tol=0.1,
    )

In [ ]:
deltaE = np.linspace(-3, 0, 100)
spectral_2d = spectral_along_path(model, band_path, deltaE, eta=0.1)
plot_spectral_path(spectral_2d, band_path, deltaE)
plt.show()

In [ ]:
curved_path = arpes_path(band_path, g_vec, Ek=21.2, V0=15)
spectral_curved = spectral_along_path(model, curved_path, deltaE, eta=0.1)
plot_spectral_path(spectral_curved, band_path, deltaE)
plt.show()

In [ ]:
deltaE = np.linspace(0, -3, 300)
v0_results = spectral_arpes_path_sweep(
    model, band_path, g_vec,
    photon_energy=21.2, v0_list=[15],
    omega_list=deltaE, eta=0.1,
)
for v0, spectral_2d in v0_results.items():
    output_dir = results_path("v0_sweep", "figs", "arpes_path", "k_400")
    output_dir.mkdir(parents=True, exist_ok=True)
    plot_spectral_path(
        spectral_2d, band_path, deltaE,
        title=rf"$V_0 = {v0}$ eV",
        save=output_dir / f"V0_{v0}.png",
    )
    plt.show()